In [1]:
# import the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score
import warnings
from sklearn.exceptions import DataConversionWarning


In [2]:
warnings.filterwarnings("ignore", category=UserWarning, 
                        module="sklearn.preprocessing._encoders")

In [3]:
def build_pipeline(model_name, path):
    # load the dataset into pandas dataframe

    print("Loading features...")
    df = pd.read_csv(path)

    numerical = list(df.select_dtypes(include=['int64', 'float64']).columns)
    if 'default_flag' in numerical:
        numerical.remove('default_flag')
        
    categorical = list(df.select_dtypes(include=['str']).columns)

    categorical_preprocessor = Pipeline(steps=[
        ("cat", OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
    ])

    numerical_preprocessor = Pipeline(steps=[
        ("numeric", SimpleImputer(strategy='median')),
        ("scaler", StandardScaler())
    ]) 

    preprocessor = ColumnTransformer(transformers=[
        ('cat', categorical_preprocessor, categorical),
        ('num', numerical_preprocessor, numerical)
    ])

    # Full ML pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model_name)
    ])
    
    return pipeline


    



In [4]:
def tune_model(model_name, X, y, params=None, cv=5, scoring="roc_auc"):
    path = '../data/features/loan_features.csv'
    
    # Model registry
    MODEL_REGISTRY = {
        "LogisticRegression": LogisticRegression(max_iter=2000),
        "RandomForest": RandomForestClassifier(random_state=42, class_weight='balanced'),
        "xgboost": xgb.XGBClassifier(random_state=42, scale_pos_weight=4)
    }
    
    # Default params for each model
    DEFAULT_PARAMS = {
        "LogisticRegression": {'model__C': [0.1, 0.3, 1.0]},
        "RandomForest": {'model__n_estimators': [100, 200], 'model__max_depth': [6, 10]},
        "xgboost": {'model__n_estimators': [200, 500], 'model__max_depth': [4, 6]}
    }
    
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model: {model_name}. Choose: {list(MODEL_REGISTRY.keys())}")
    
    # Build & tune
    pipeline = build_pipeline(MODEL_REGISTRY[model_name], path)
    parameters = params or DEFAULT_PARAMS[model_name]
    
    print(f"Tuning {model_name}...")
    grid = GridSearchCV(pipeline, param_grid=parameters, cv=cv, scoring=scoring, n_jobs=-1)
    grid.fit(X, y)
    
    return {
        'best_estimator': grid.best_estimator_,
        'best_score': grid.best_score_, 
        'best_parameters': grid.best_params_,
        'cv_results': grid.cv_results_
    }

In [5]:
df = pd.read_csv('../data/features/loan_features.csv')

numerical = list(df.select_dtypes(include=['int64', 'float64']).columns)
categorical = list(df.select_dtypes(include=['str']).columns)

X = df[numerical+categorical]
y = df['default_flag']

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)


tuned = tune_model("LogisticRegression", X_train, y_train, 
           params={'model__C': [0.1, 0.3, 0.5, 1]}, cv=5, scoring="roc_auc")

Loading features...
Tuning LogisticRegression...


In [6]:
score = tuned['best_score']
print(f'train_auc: {score}')

train_auc: 0.7115716031419262


In [7]:
best_model = tuned['best_estimator']
y_pred = best_model.predict_proba(X_test)[:, 1]
print(f'test_auc: {roc_auc_score(y_test, y_pred)}')

test_auc: 0.7160200727697225


In [10]:
tuned['cv_results']

{'mean_fit_time': array([4.80272851, 3.94527378, 3.23468432, 3.01288052]),
 'std_fit_time': array([0.34256349, 0.40200772, 0.20713946, 0.22113091]),
 'mean_score_time': array([0.2618835 , 0.17803135, 0.14838924, 0.10247478]),
 'std_score_time': array([0.06397655, 0.0414298 , 0.02424965, 0.01409247]),
 'param_model__C': masked_array(data=[0.1, 0.3, 0.5, 1.0],
              mask=[False, False, False, False],
        fill_value=1e+20),
 'params': [{'model__C': 0.1},
  {'model__C': 0.3},
  {'model__C': 0.5},
  {'model__C': 1}],
 'split0_test_score': array([0.72010746, 0.72017877, 0.72024942, 0.72026701]),
 'split1_test_score': array([0.72013683, 0.71926293, 0.71892102, 0.7186121 ]),
 'split2_test_score': array([0.70445889, 0.70392895, 0.70377985, 0.70377839]),
 'split3_test_score': array([0.70249685, 0.70249082, 0.70243222, 0.70246261]),
 'split4_test_score': array([0.71065799, 0.71038331, 0.71028607, 0.71013604]),
 'mean_test_score': array([0.7115716 , 0.71124895, 0.71113372, 0.71105123])